In [1]:
!pip install gensim

import numpy as np
import torch
from collections import Counter
from gensim.models import KeyedVectors
import gensim.downloader as api



/usr/local/lib/python3.12/dist-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [2]:
import pandas as pd
df=pd.read_csv("/kaggle/input/datasets/ldhhieu18/final-data/laodong_all_news_with_split.csv")

In [3]:
# =========================================
# CHUẨN BỊ DỮ LIỆU
# =========================================
print("\n" + "="*80)
print("PREPARING DATA FROM 'content' COLUMN (TRAIN ONLY)")
print("="*80)

# Lọc chỉ lấy tập train (is_test == 0)
df_train = df[df['is_test'] == 0].copy()
print(f"Total rows: {len(df)}")
print(f"Train rows (is_test=0): {len(df_train)}")
print(f"Test rows  (is_test=1): {len(df) - len(df_train)}")

texts = df_train['content'].tolist()
sentences = [text.split() for text in texts if isinstance(text, str) and len(text.strip()) > 0]
print(f"Example sentence: {sentences[0][:10]}")
print(f"Total sentences: {len(sentences)}")
print(f"Total tokens: {sum(len(s) for s in sentences)}")


PREPARING DATA FROM 'content' COLUMN (TRAIN ONLY)
Total rows: 6000
Train rows (is_test=0): 5100
Test rows  (is_test=1): 900
Example sentence: ['Lãi', 'suất', 'ngân', 'hàng', 'hôm', 'nay', '2.1:', 'Tăng', 'mạnh,', 'đua']
Total sentences: 5100
Total tokens: 2681617


In [4]:
# =========================================
# XÂY DỰNG VOCABULARY
# =========================================
print("\n⏳ Building vocabulary...")

word_freq = Counter()
for sentence in sentences:
    word_freq.update(sentence)

word2idx = {'<PAD>': 0, '<UNK>': 1}
idx2word = {0: '<PAD>', 1: '<UNK>'}

idx = 2
for word, freq in word_freq.most_common():
    if freq >= 2:
        word2idx[word] = idx
        idx2word[idx] = word
        idx += 1

vocab_size = len(word2idx)
print(f"Vocabulary built: {vocab_size} words")




⏳ Building vocabulary...
Vocabulary built: 28126 words


In [5]:
# =========================================
# OPTION 1: TRAIN WORD2VEC
# =========================================
from gensim.models import Word2Vec
print("\n" + "="*80)
print("TRAINING WORD2VEC (RECOMMENDED)")
print("="*80)

W2V_DIM = 300
W2V_EPOCHS = 20

print(f"Training Word2Vec...")
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=W2V_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=W2V_EPOCHS,
    seed=42
)

print("Word2Vec training completed")

# Tạo embedding matrix
embedding_matrix = np.random.normal(0, 0.1, (vocab_size, W2V_DIM))
found = 0
for word, idx in word2idx.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        found += 1

print(f"Coverage: {found}/{vocab_size} ({found/vocab_size*100:.2f}%)")

embedding_matrix = torch.FloatTensor(embedding_matrix)

# Lưu
w2v_model.save("word2vec_content.model")
torch.save(embedding_matrix, "embedding_content.pt")
torch.save(word2idx, "word2idx_content.pt")

print("Saved: word2vec_content.model, embedding_content.pt, word2idx_content.pt")
print("\nDone! Use these files in your LSTM model.")


TRAINING WORD2VEC (RECOMMENDED)
Training Word2Vec...
Word2Vec training completed
Coverage: 28124/28126 (99.99%)
Saved: word2vec_content.model, embedding_content.pt, word2idx_content.pt

Done! Use these files in your LSTM model.
